#### **Introduction to free energy sampling by Umbrella Sampling** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

Umbrella sampling (US) is perhaps the most common and one of the more robust ways of calculating free-energy landscapes using molecular dynamics simulations. Using US we bias our system to ensure sampling along a specific set of reaction coordinates. After, we un-bias our simulation to recover the underlying Potential of Mean Force (PMF) (close analog to the free-energy landscape). Special considerations must be made for:

- Sampling time
- Choice of reaction coordinate

There are many great tutorials available on the web for learning how to pratically set up US simulations in essentially every MD engine. For reference, I link some of these here:
AMBER: http://ambermd.org/tutorials/advanced/tutorial17/
GROMACS: http://www.mdtutorials.com/gmx/umbrella/index.html
NAMD: http://www.ks.uiuc.edu/Training/Tutorials/science/channel/channel-tut.pdf


The intent of this Jupyter Notebook is **not** to teach the reader how to set up US simulations. The above tutorials already do a better job than I could. Instead, I hope that by playing around with a model system the reader will develop an intuitive understanding of how US works and to take their understanding beyond that of a black box operation. 

Below, we introduce a model potential that a particle in a box can be subjected to. This model potential is based off:
http://msmbuilder.org/3.8.0/examples/tICA-vs-PCA.html. The potential is symmetric, and contains two wells in the X direction.

In [ ]:
# --- Plotly is a powerful library for interactive plots ---
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

xx, yy = np.meshgrid(np.linspace(-2,2), np.linspace(-6,6))
zz = 0 # We can only visualize so many dimensions
ww = 5 * (xx-1)**2 * (xx+1)**2 + yy**2 + zz**2

fig = go.Figure(data = go.Contour(x=np.linspace(-2,2), y=np.linspace(-6,6), z=ww, line_width=2,
                                 contours=dict(start=0, end=25, size=2)))
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
# --- install openmm with conda: conda install -c conda-forge openmm ---
import openmm as mm

# --- define our simulation using OpenMM ---
def propagate(n_steps=10000):
    system = mm.System()
    system.addParticle(1)
    force = mm.CustomExternalForce('5*(x-1)^2*(x+1)^2 + y^2 + z^2')
    force.addParticle(0, [])
    system.addForce(force)
    integrator = mm.LangevinIntegrator(500, 1, 0.02)
    context = mm.Context(system, integrator)
    context.setPositions([[0, 0, 0]])
    context.setVelocitiesToTemperature(500)
    x = np.zeros((n_steps, 3))
    for i in range(n_steps):
        x[i] = (context.getState(getPositions=True)
                .getPositions(asNumpy=True)
                ._value)
        integrator.step(1)
    return pd.DataFrame(x,columns=['X', 'Y', 'Z'])

In [ ]:
# --- Perform a simulation ---
trajectory = propagate(10000)

In [ ]:
# --- take a look at our data --- 
trajectory

In [ ]:
# --- Plot our X, Y, and Z coordinates versus time. ---
fig = px.line(trajectory, y=["X", "Y", "Z"])
fig.show()

What do you notice about the data in the x, y, and z directions?

In [ ]:
# --- Project our X & Y data on the landscape ---

fig = go.Figure(data = go.Contour(x=np.linspace(-2,2), y=np.linspace(-6,6), z=ww, line_width=2,
                                 contours=dict(start=0, end=25, size=2), colorscale="greys")) 

fig.add_trace(
    go.Scatter(
        x=trajectory["X"],
        y=trajectory["Y"],
        mode="markers",
        opacity=0.5))
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
# --- Let's plot a histogram to see what's up ---

hist_fig = px.histogram(trajectory, x="X",
                        marginal="box", histnorm='probability')
hist_fig.show()

Now that we understand how our simulation behaves on the underlying energy landscape, we are ready to bias our simulation. One convenient (though not the only) way to bias our system is with a **harmonic potential.** Essentially, we will couple our simulation to come chosen value $x_{o}$ and penalize the energy of the system for deviations away from this equilibrium value.

Importantly, we are not only simulating the harmonic potential. Instead, we are simulating the resultant potential from summing the harmonic potential with our unerlying energy landscape. In this toy example we know *a priori* what the underlying landscape looks like. In real-world examples, you are blind to the unerlying energy landscape.

In [ ]:
def propagate_biased_X(n_steps=10000, x0=0, k=100):
    system = mm.System()
    system.addParticle(1)
    force = mm.CustomExternalForce('5*(x-1)^2*(x+1)^2 + y^2 + z^2 + k*(x-x0)^2')
    force.addGlobalParameter("k", k) 
    # note that if k == 0, then we run an unbiased simulation
    force.addGlobalParameter("x0", x0)
    force.addParticle(0, [])
    system.addForce(force)
    integrator = mm.LangevinIntegrator(500, 1, 0.02)
    context = mm.Context(system, integrator)
    context.setPositions([[0, 0, 0]])
    context.setVelocitiesToTemperature(500)
    x = np.zeros((n_steps, 3))
    for i in range(n_steps):
        x[i] = (context.getState(getPositions=True)
                .getPositions(asNumpy=True)
                ._value)
        integrator.step(1)
    return pd.DataFrame(x,columns=['X', 'Y', 'Z'])

In [ ]:
# --- perform the biased simulation --- 
biased_traj = propagate_biased_X(10000)

In [ ]:
fig = go.Figure(data = go.Contour(x=np.linspace(-2,2), y=np.linspace(-6,6), z=ww, line_width=2,
                                 contours=dict(start=0, end=25, size=2), colorscale="greys")) 
fig.add_trace(
    go.Scatter(
        x=biased_traj["X"],
        y=biased_traj["Y"],
        mode="markers",
        opacity=0.5
    ))

# --- Show where our bias is located ---
fig.add_trace(
    go.Scatter(
        x=[0,0],
        y=[-4,4],
        mode="lines",
        line=dict(color='cyan', width=4)))
fig.update_layout(width=800, height=600)

fig.show()

In [ ]:
# --- Again, plot the histogram ---
hist_fig = px.histogram(biased_traj, x="X",
                        marginal="box", histnorm='probability')
hist_fig.update_xaxes(range=[-1, 1])
hist_fig.show()

By changing the spring constant $k$, we can change how tight our simulation is coupled to our chosen $x_{o}$.

In [ ]:
biased_traj_tight = propagate_biased_X(10000, 0, 500)
hist_fig = px.histogram(biased_traj_tight, x="X",
                        marginal="box", histnorm='probability')
hist_fig.update_xaxes(range=[-1, 1])
hist_fig.show()

By changing the equilibrium position $x_{o}$, we can change where our coupling is centered.

In [ ]:
biased_traj_right = propagate_biased_X(10000, 0.4, 500)
hist_fig = px.histogram(biased_traj_right, x="X",
                        marginal="box", histnorm='probability')
hist_fig.update_xaxes(range=[-1, 1])
hist_fig.show()

In [ ]:
fig = go.Figure(
    data = go.Contour(
            x=np.linspace(-2,2), 
            y=np.linspace(-6,6), 
            z=ww, 
            line_width=2,
            contours=dict(start=0, end=25, size=2), 
            colorscale="greys"
    ),
    layout=go.Layout(
            width=900,
            height=700
    )
) 

# traj set at x0=0 with k=100
fig.add_trace(
    go.Scatter(
        x=biased_traj["X"],
        y=biased_traj["Y"],
        mode="markers",
        opacity=0.5
    ))
# traj set at x0=0 with k=500
fig.add_trace(
    go.Scatter(
        x=biased_traj_tight["X"],
        y=biased_traj_tight["Y"],
        mode="markers",
        opacity=0.5
    ))
# traj set at x0=0.4 with k=500
fig.add_trace(
    go.Scatter(
        x=biased_traj_right["X"],
        y=biased_traj_right["Y"],
        mode="markers",
        opacity=0.5
    ))

fig.show()

Great! So now we know how to couple our simulation with a **bias potential.** Let's return to the idea of umbrella sampling. Ideally, we would like to recover the underlying energy landscape from our simulations. The reader should recall that in lecture 1 we discussed the relationship between the **energy of a configuration** and the **probability of observing that configuration.** 

We should be able to relate the probability of sampling a state to the energy of that state. *What is the precise relationship?*

So, let's return to the unbiased simulation and plot the $(X,Y)$ probability density of our particle.

In [ ]:
fig = px.density_contour(trajectory, x="X", y="Y")
fig.update_traces(contours_coloring="fill", contours_showlabels=False)
fig.show()

In statistical mechanics, the free energy (or potential of mean force), $PMF(r)$ is governed by the Boltzmann distribution. That is: $$PMF(r) = -k_{B}T\ln(P(r)) + C$$ 

Where $C$ is an arbitrary constant. We typically neglect $C$ since we usually only care about changes in free energy. 

In [ ]:
import numpy as np
kT = 4.157  # approx value for 500 K in kJ/mol

# --- Calculate a histogram ---
counts, bin_edges = np.histogram(trajectory["X"], bins=50, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# --- Calculate the energy from the negative log of probabilities --- 
energy = -kT * np.log(counts)
energy -= np.min(energy) # Shift so minimum is 0

# --- Plot ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=bin_centers, y=energy))
fig.show()

Great! So just from the equilibrium sampling alone we can get an estimate of the free energy profile of our simulation. 

How do we unbias our simulation once we've added a harmonic potential?

In [ ]:
biased_traj_right = propagate_biased_X(50000, 0.4, 500)

# --- Calculate a histogram ---
counts, bin_edges = np.histogram(biased_traj_right["X"], bins=50, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# --- Calculate the energy from the negative log of probabilities --- 
biased_energy = -kT * np.log(counts)
x0, k = 0.4, 500 # match this to the bias in the original simulation
v_bias = k * (bin_centers - x0)**2 # calculate bias at each bin center
reweighted_energy = biased_energy - v_bias
reweighted_energy -= np.nanmin(reweighted_energy)

# --- Plot ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=bin_centers, y=biased_energy, name="Biased Energy (Observed)"))
fig.add_trace(go.Scatter(x=bin_centers, y=reweighted_energy, name="Reweighted Energy"))
fig.show()

What can you tell about the shape and discontinuities of our energy profiles? Where is our estimate of the free energy more accurate? Less accurate?